# CLV Scoring, Validation & Segmentation

Combines purchase probability with spend-tier expected revenue to compute CLV, validates against the temporal holdout, and segments customers into actionable tiers.

**Part A:** Expected revenue + combined CLV
**Part B:** Temporal holdout validation
**Part C:** Customer segmentation + campaign ROI

- **Input:** `data/processed/stage1_scored.csv` (customers with `p_purchase`)
- **Output:** `data/processed/clv_final.csv` (customers with CLV + segments)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import average_precision_score, roc_auc_score, mean_absolute_error

## Part A: Expected Revenue + Combined CLV

Estimates **expected revenue per buyer** using spend-tier averages, then combines with Stage 1 purchase probability.

**Why not individual regression?** Individual-level revenue prediction is typically noisy — the features that predict *who* buys do not necessarily predict *how much* they spend. Instead, we use historical spend tiers as a natural revenue proxy.

### 1. Load Stage 1 Output

In [ ]:
df = pd.read_csv('../data/processed/stage1_scored.csv')
print(f"Loaded {len(df):,} customers")
print(f"p_purchase range: [{df['p_purchase'].min():.4f}, {df['p_purchase'].max():.4f}]")
print(f"p_purchase mean:  {df['p_purchase'].mean():.4f}")

buyers = df[df['actual_holdout_transactions'] > 0].copy()
non_buyers = df[df['actual_holdout_transactions'] == 0].copy()
print(f"\nHoldout buyers:     {len(buyers):,} ({len(buyers)/len(df):.1%})")
print(f"Holdout non-buyers: {len(non_buyers):,} ({len(non_buyers)/len(df):.1%})")
print(f"\nHoldout period: 2011-06-09 to 2011-12-09 (~183 days)")

### 2. Spend Tier Construction

In [ ]:
# Create spend tiers using terciles of monetary_value (all customers)
df['spend_tier'] = pd.qcut(
    df['monetary_value'], q=3,
    labels=['Low Spend', 'Mid Spend', 'High Spend']
)

print("Spend tier thresholds (monetary_value):")
for tier in ['Low Spend', 'Mid Spend', 'High Spend']:
    tier_data = df[df['spend_tier'] == tier]['monetary_value']
    print(f"  {tier}: ${tier_data.min():.2f} -- ${tier_data.max():.2f} ({len(tier_data):,} customers)")

In [ ]:
# Compute average holdout revenue PER BUYER within each tier
buyers_with_tier = df[df['actual_holdout_transactions'] > 0].copy()

tier_revenue = buyers_with_tier.groupby('spend_tier').agg(
    n_buyers           = ('user_id', 'count'),
    avg_holdout_rev    = ('actual_holdout_revenue', 'mean'),
    median_holdout_rev = ('actual_holdout_revenue', 'median'),
    total_holdout_rev  = ('actual_holdout_revenue', 'sum'),
).round(2)

print("=== Tier-Level Revenue (holdout buyers only) ===")
print(tier_revenue.to_string())
print(f"\nOverall avg holdout revenue per buyer: ${buyers_with_tier['actual_holdout_revenue'].mean():.2f}")

In [ ]:
# KEY CHECK: Do spend tiers produce meaningful revenue differentiation?
tier_avgs = tier_revenue['avg_holdout_rev']
tier_range = tier_avgs.max() - tier_avgs.min()
tier_cv = tier_avgs.std() / tier_avgs.mean() if tier_avgs.mean() > 0 else 0

print("=== Tier Differentiation Check ===")
for tier in ['Low Spend', 'Mid Spend', 'High Spend']:
    print(f"  {tier}: ${tier_avgs[tier]:.2f} avg holdout revenue per buyer")
print(f"\n  Range: ${tier_range:.2f}")
print(f"  CV (coefficient of variation): {tier_cv:.3f}")

if tier_cv < 0.05:
    print("\n  ** Tier averages are nearly flat (CV < 5%). **")
    print("  This means monetary_value does NOT strongly predict holdout spend.")
    print("  CLV differentiation will come primarily from P(purchase), not E[revenue].")
    print("  The tier approach still works -- it just means CLV ~ p_purchase x constant.")
else:
    print(f"\n  Tiers show meaningful differentiation (CV = {tier_cv:.1%}).")
    print("  Higher historical spenders do spend more in holdout -- tiers add signal.")

In [ ]:
# Visualize revenue by tier
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#9E9E9E', '#4CAF50', '#2196F3']
tiers = ['Low Spend', 'Mid Spend', 'High Spend']

# Average revenue per buyer by tier
avg_revs = [tier_revenue.loc[t, 'avg_holdout_rev'] for t in tiers]
axes[0].bar(tiers, avg_revs, color=colors, edgecolor='white')
axes[0].set_title('Avg Holdout Revenue per Buyer by Tier')
axes[0].set_ylabel('Avg Revenue ($)')
for i, v in enumerate(avg_revs):
    axes[0].text(i, v + max(avg_revs) * 0.02, f'${v:.0f}', ha='center', fontsize=10)

# Number of buyers per tier
n_buyers_list = [tier_revenue.loc[t, 'n_buyers'] for t in tiers]
axes[1].bar(tiers, n_buyers_list, color=colors, edgecolor='white')
axes[1].set_title('Number of Holdout Buyers by Tier')
axes[1].set_ylabel('Buyers')
for i, v in enumerate(n_buyers_list):
    axes[1].text(i, v + max(n_buyers_list) * 0.02, f'{int(v):,}', ha='center', fontsize=10)

plt.suptitle('Spend Tier Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Map Expected Revenue to All Customers

In [ ]:
# Create tier -> expected revenue mapping
tier_avg_map = tier_revenue['avg_holdout_rev'].to_dict()
print("Tier -> Expected Revenue mapping:")
for tier, rev in tier_avg_map.items():
    print(f"  {tier}: ${rev:.2f}")

# Map to all customers
df['expected_revenue_if_purchase'] = df['spend_tier'].map(tier_avg_map)

print(f"\nMapped to all {len(df):,} customers")
print(f"Null values: {df['expected_revenue_if_purchase'].isna().sum()}")

### 4. Combined CLV

The holdout window runs from 2011-06-09 to 2011-12-09, which is **183 days**.

- `clv_180d = p_purchase x expected_revenue_if_purchase` (holdout-period CLV)
- `clv_12m = clv_180d x (365 / 183)` (annualized estimate via linear extrapolation)

In [ ]:
HOLDOUT_DAYS = 183  # 2011-06-09 to 2011-12-09

# CLV = P(purchase) x E[revenue | purchase]
df['expected_revenue_if_purchase'] = pd.to_numeric(
    df['expected_revenue_if_purchase'], errors='coerce'
)
df['clv_180d'] = df['p_purchase'] * df['expected_revenue_if_purchase']

# Scale to 12-month estimate using actual holdout length
df['clv_12m'] = df['clv_180d'] * (365 / HOLDOUT_DAYS)

print("=== Combined CLV Distribution ===")
print(f"CLV (180d): ${df['clv_180d'].mean():.2f} avg, ${df['clv_180d'].median():.2f} median")
print(f"CLV (12m):  ${df['clv_12m'].mean():.2f} avg, ${df['clv_12m'].median():.2f} median")
print(f"\nComponents:")
print(f"  P(purchase):     {df['p_purchase'].mean():.4f} avg")
print(f"  E[rev|purchase]: ${df['expected_revenue_if_purchase'].mean():.2f} avg")

print(f"\nCLV (180d) percentiles:")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  P{p}: ${df['clv_180d'].quantile(p/100):.2f}")

In [ ]:
# CLV distribution histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['clv_180d'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('CLV (180-day) Distribution')
axes[0].set_xlabel('CLV ($)')
axes[0].axvline(
    df['clv_180d'].median(), color='red', linestyle='--',
    label=f"Median: ${df['clv_180d'].median():.2f}"
)
axes[0].legend()

# CLV by spend tier
tier_clv = df.groupby('spend_tier')['clv_180d'].mean()
axes[1].bar(tiers, [tier_clv[t] for t in tiers], color=colors, edgecolor='white')
axes[1].set_title('Avg CLV (180d) by Spend Tier')
axes[1].set_ylabel('Avg CLV ($)')
for i, t in enumerate(tiers):
    axes[1].text(i, tier_clv[t] + tier_clv.max() * 0.02,
                f'${tier_clv[t]:.2f}', ha='center', fontsize=10)

plt.suptitle('Combined CLV Analysis', fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Sanity Checks

In [ ]:
# Check 1: One-time buyers should have lower CLV than repeat buyers
clv_onetime = df.loc[df['frequency'] == 0, 'clv_180d'].mean()
clv_repeat  = df.loc[df['frequency'] >= 1, 'clv_180d'].mean()
print(f"One-time buyer avg CLV (180d): ${clv_onetime:.2f}")
print(f"Repeat buyer avg CLV (180d):   ${clv_repeat:.2f}")
print(f"Check: one-time < repeat: {'PASS' if clv_onetime < clv_repeat else 'FAIL'}")

# Check 2: Top 20% should capture disproportionate share of predicted CLV
top20_n = int(len(df) * 0.2)
top20_clv = df.nlargest(top20_n, 'clv_180d')['clv_180d'].sum()
total_clv = df['clv_180d'].sum()
top20_pct = top20_clv / total_clv
print(f"\nTop 20% customers account for {top20_pct:.1%} of total predicted CLV")

# Check 3: Total predicted vs actual holdout revenue
total_pred = df['clv_180d'].sum()
total_actual = df['actual_holdout_revenue'].sum()
print(f"\nTotal predicted (180d): ${total_pred:,.0f}")
print(f"Total actual (holdout): ${total_actual:,.0f}")
print(f"Ratio: {total_pred / total_actual:.2f}")

---
## Part B: Temporal Holdout Validation

Part A computed CLV for all 4,918 customers. This section validates the two-stage model against the 183-day holdout window (2011-06-09 to 2011-12-09).

1. **Purchase propensity ranking quality** (decile analysis)
2. **Revenue prediction accuracy** for holdout buyers
3. **CLV lift curve** — does ranking by predicted CLV capture actual holdout revenue?
4. **Revenue calibration** — total predicted vs actual

### 1. Purchase Propensity Validation

How well does `p_purchase` rank customers by likelihood of holdout purchase?

**Note:** Full-dataset metrics include training data and overestimate generalization. See notebook 01 for test-set metrics (PR-AUC: 0.8677, ROC-AUC: 0.8483).

In [ ]:
y_true = (df['actual_holdout_transactions'] > 0).astype(int)
y_prob = df['p_purchase']

pr_auc  = average_precision_score(y_true, y_prob)
roc_auc = roc_auc_score(y_true, y_prob)
baseline = y_true.mean()

print("=== Stage 1: Purchase Propensity (FULL DATASET) ===")
print(f"PR-AUC:   {pr_auc:.4f}  (baseline: {baseline:.4f}, lift: {pr_auc / baseline:.1f}x)")
print(f"ROC-AUC:  {roc_auc:.4f}")
print()
print("Note: Full-dataset metrics include training data and overestimate generalization.")
print("See notebook 01 for test-set metrics: PR-AUC 0.8677, ROC-AUC 0.8483.")

In [ ]:
# Decile analysis: bin customers by p_purchase, show actual purchase rate
df['propensity_decile'] = pd.qcut(
    df['p_purchase'], 10, labels=False, duplicates='drop'
) + 1

decile_stats = df.groupby('propensity_decile').agg(
    n_customers    = ('user_id', 'count'),
    actual_rate    = ('purchased_in_holdout', 'mean'),
    avg_p_purchase = ('p_purchase', 'mean'),
    total_holdout_rev = ('actual_holdout_revenue', 'sum'),
).round(4)

print("=== Propensity Decile Analysis (Full Dataset) ===")
print(decile_stats.to_string())

top_decile_rate = decile_stats.iloc[-1]['actual_rate']
bottom_decile_rate = decile_stats.iloc[0]['actual_rate']
decile_lift = top_decile_rate / max(bottom_decile_rate, 0.001)
print(f"\nTop decile purchase rate:    {top_decile_rate:.1%}")
print(f"Bottom decile purchase rate: {bottom_decile_rate:.1%}")
print(f"Lift (top vs bottom):        {decile_lift:.1f}x")

In [ ]:
# Plot purchase rate by propensity decile
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(decile_stats.index, decile_stats['actual_rate'],
       color='steelblue', edgecolor='white')
ax.axhline(y=baseline, color='red', linestyle='--',
           label=f'Overall rate ({baseline:.1%})')
ax.set_xlabel('Propensity Decile (1=lowest, 10=highest)')
ax.set_ylabel('Actual Purchase Rate')
ax.set_title('Purchase Rate by Propensity Decile')
ax.set_xticks(decile_stats.index)
ax.legend()
plt.tight_layout()
plt.show()

### 2. Revenue Prediction Validation

For customers who actually purchased in the holdout, how well does `expected_revenue_if_purchase` match actual revenue at the tier level?

In [ ]:
buyers = df[df['actual_holdout_transactions'] > 0].copy()

# Compare tier-level predictions vs actuals
tier_validation = buyers.groupby('spend_tier').agg(
    n_buyers          = ('user_id', 'count'),
    predicted_avg_rev = ('expected_revenue_if_purchase', 'mean'),
    actual_avg_rev    = ('actual_holdout_revenue', 'mean'),
    actual_median_rev = ('actual_holdout_revenue', 'median'),
).round(2)
tier_validation['error_pct'] = (
    (tier_validation['predicted_avg_rev'] - tier_validation['actual_avg_rev'])
    / tier_validation['actual_avg_rev'] * 100
).round(1)

print("=== Tier-Level Revenue Validation (holdout buyers) ===")
print(tier_validation.to_string())

# Overall MAE for buyers
mae = mean_absolute_error(
    buyers['actual_holdout_revenue'],
    buyers['expected_revenue_if_purchase']
)
print(f"\nOverall MAE (buyers only): ${mae:.2f}")
print(f"Mean actual:    ${buyers['actual_holdout_revenue'].mean():.2f}")
print(f"Mean predicted: ${buyers['expected_revenue_if_purchase'].mean():.2f}")

### 3. CLV Lift Curve

In [ ]:
# Sort by predicted CLV descending
df_ranked = df.sort_values('clv_180d', ascending=False).reset_index(drop=True)
total_actual = df_ranked['actual_holdout_revenue'].sum()

# Cumulative revenue capture
df_ranked['cum_actual_rev'] = df_ranked['actual_holdout_revenue'].cumsum()
df_ranked['cum_actual_pct'] = df_ranked['cum_actual_rev'] / total_actual * 100
df_ranked['customer_pct']   = (df_ranked.index + 1) / len(df_ranked) * 100

# Key checkpoints
print("=== CLV Lift Curve Checkpoints ===")
for pct in [10, 20, 30, 50]:
    idx = int(len(df_ranked) * pct / 100) - 1
    capture = df_ranked.iloc[idx]['cum_actual_pct']
    print(f"Top {pct}% by CLV captures {capture:.1f}% of actual holdout revenue")

top20_capture = df_ranked.iloc[int(len(df_ranked) * 0.2) - 1]['cum_actual_pct']

In [ ]:
# Lift curve plot
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(df_ranked['customer_pct'], df_ranked['cum_actual_pct'],
        color='steelblue', linewidth=2, label='Two-Stage CLV Model')
ax.plot([0, 100], [0, 100], 'k--', linewidth=1, label='Random (no model)')

# Annotate top 20% capture
ax.axvline(x=20, color='red', linestyle=':', alpha=0.5)
ax.annotate(f'Top 20% captures {top20_capture:.1f}%',
            xy=(20, top20_capture), xytext=(35, top20_capture - 8),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red')

ax.set_xlabel('% of Customers (ranked by predicted CLV)')
ax.set_ylabel('% of Actual Holdout Revenue Captured')
ax.set_title('CLV Lift Curve: Predicted CLV vs Actual Holdout Revenue')
ax.legend(loc='lower right')
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

### 4. Revenue Calibration

In [ ]:
total_pred_180d = df['clv_180d'].sum()
total_actual    = df['actual_holdout_revenue'].sum()

print("=== Revenue Calibration ===")
print(f"Total predicted CLV (180d):    ${total_pred_180d:,.0f}")
print(f"Total actual holdout revenue:  ${total_actual:,.0f}")
print(f"Ratio (predicted/actual):      {total_pred_180d / total_actual:.3f}")
print(f"")
print(f"Mean predicted CLV (180d):     ${df['clv_180d'].mean():.2f}")
print(f"Mean actual holdout revenue:   ${df['actual_holdout_revenue'].mean():.2f}")

### 5. Validation Summary

In [ ]:
print("=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print(f"")
print(f"Stage 1 -- Purchase Propensity:")
print(f"  PR-AUC (full dataset):  {pr_auc:.4f} ({pr_auc / baseline:.1f}x over baseline)")
print(f"  ROC-AUC (full dataset): {roc_auc:.4f}")
print(f"  PR-AUC (test-set):      0.8677  (from notebook 01)")
print(f"  ROC-AUC (test-set):     0.8483  (from notebook 01)")
print(f"  Decile lift:            {decile_lift:.1f}x (top vs bottom decile)")
print(f"")
print(f"Stage 2 -- Revenue (tier-based):")
print(f"  MAE (buyers only):      ${mae:.2f}")
print(f"  Mean actual (buyers):   ${buyers['actual_holdout_revenue'].mean():.2f}")
print(f"")
print(f"Combined CLV:")
print(f"  Top 20% capture:        {top20_capture:.1f}% of holdout revenue")
print(f"  Revenue ratio:          {total_pred_180d / total_actual:.3f}")
print("=" * 60)

---
## Part C: Customer Segmentation & Campaign ROI

The model is validated. This section operationalizes CLV predictions into actionable business outputs:

1. **4-tier customer segmentation** based on CLV and P(purchase)
2. **Segment profiles** with size, avg CLV, and predicted revenue
3. **Campaign ROI allocation table** for budget optimization

### 1. 4-Tier Customer Segmentation

| Segment | Definition | Recommended Action |
|---------|------------|--------------------|
| **High Value** | Top 20% CLV | No discounts -- protect margin |
| **At-Risk** | p_purchase < 0.20 (any CLV band) | Win-back campaign |
| **Growing** | Middle 40% CLV + p_purchase >= 0.20 | Personalized offers |
| **Low Value** | Bottom 40% CLV + p_purchase >= 0.20 | Email-only, minimal spend |

**Note:** At-Risk is checked across all CLV bands (except High Value) and takes priority over Growing/Low Value. The threshold of 0.20 is well below the ~52% base purchase rate, flagging customers with genuinely low engagement.

In [ ]:
P_PURCHASE_ATRISK = 0.20

clv_top20_threshold    = df['clv_12m'].quantile(0.80)
clv_bottom40_threshold = df['clv_12m'].quantile(0.40)

print(f"Segmentation thresholds:")
print(f"  Top 20% (High Value):   CLV > ${clv_top20_threshold:.2f}")
print(f"  Middle 40% (Growing):   ${clv_bottom40_threshold:.2f} < CLV <= ${clv_top20_threshold:.2f}")
print(f"  Bottom 40% (Low Value): CLV <= ${clv_bottom40_threshold:.2f}")
print(f"  At-Risk:                p_purchase < {P_PURCHASE_ATRISK} (overrides Growing/Low Value)")

def assign_segment(row):
    if row['clv_12m'] > clv_top20_threshold:
        return 'High Value'
    elif row['p_purchase'] < P_PURCHASE_ATRISK:
        return 'At-Risk'
    elif row['clv_12m'] > clv_bottom40_threshold:
        return 'Growing'
    else:
        return 'Low Value'

df['segment'] = df.apply(assign_segment, axis=1)

segment_counts = df['segment'].value_counts()
print(f"\nSegment distribution:")
for seg, cnt in segment_counts.items():
    print(f"  {seg:15s}: {cnt:,} customers ({cnt/len(df):.1%})")

### 2. Segment Profiles

In [ ]:
seg_order = ['High Value', 'Growing', 'At-Risk', 'Low Value']

seg_profile = df.groupby('segment').agg(
    n_customers        = ('user_id', 'count'),
    avg_clv_12m        = ('clv_12m', 'mean'),
    median_clv_12m     = ('clv_12m', 'median'),
    total_pred_revenue = ('clv_12m', 'sum'),
    avg_p_purchase     = ('p_purchase', 'mean'),
    avg_frequency      = ('frequency', 'mean'),
    avg_monetary       = ('monetary_value', 'mean'),
).round(2)

seg_profile['revenue_share'] = (
    seg_profile['total_pred_revenue']
    / seg_profile['total_pred_revenue'].sum() * 100
).round(1)

seg_profile = seg_profile.reindex(
    [s for s in seg_order if s in seg_profile.index]
)

print("=== Segment Profile ===")
print(seg_profile.to_string())

In [ ]:
# Segment visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = {
    'High Value': '#2196F3',
    'Growing':    '#4CAF50',
    'At-Risk':    '#FF5722',
    'Low Value':  '#9E9E9E',
}
segs = [s for s in seg_order if s in seg_profile.index]

# Customer count by segment
counts = [seg_profile.loc[s, 'n_customers'] for s in segs]
bar_colors = [colors[s] for s in segs]
axes[0].bar(segs, counts, color=bar_colors, edgecolor='white')
axes[0].set_title('Customer Count by Segment')
axes[0].set_ylabel('Customers')
axes[0].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)
for i, v in enumerate(counts):
    axes[0].text(i, v + max(counts) * 0.01, f'{v:,}',
                ha='center', fontsize=9)

# Predicted revenue by segment
revenues = [seg_profile.loc[s, 'total_pred_revenue'] for s in segs]
axes[1].bar(segs, revenues, color=bar_colors, edgecolor='white')
axes[1].set_title('Predicted 12-Month Revenue by Segment')
axes[1].set_ylabel('Predicted Revenue ($)')
axes[1].yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'${x:,.0f}')
)
for i, (v, s) in enumerate(zip(revenues, segs)):
    share = seg_profile.loc[s, 'revenue_share']
    axes[1].text(i, v + max(revenues) * 0.01, f'{share:.1f}%',
                ha='center', fontsize=9)

plt.suptitle('CLV Segment Overview', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3. Campaign ROI Allocation Table

| Segment | Budget/Customer | Conversion Rate | Rationale |
|---------|----------------|-----------------|----------|
| High Value | $0 (organic) | 40% | Retain naturally; discounts destroy margin |
| Growing | $15 | 25% | Personalized offer justified by growth potential |
| At-Risk | $10 | 15% | Win-back campaign; lower success rate |
| Low Value | $2 (email) | 5% | Email only; minimum investment |

In [ ]:
campaign_params = {
    'High Value': {
        'budget_per_customer': 0,
        'conversion_rate': 0.40,
        'action': 'VIP loyalty -- no campaign',
    },
    'Growing': {
        'budget_per_customer': 15,
        'conversion_rate': 0.25,
        'action': 'Personalized offer',
    },
    'At-Risk': {
        'budget_per_customer': 10,
        'conversion_rate': 0.15,
        'action': 'Win-back campaign',
    },
    'Low Value': {
        'budget_per_customer': 2,
        'conversion_rate': 0.05,
        'action': 'Email only',
    },
}

roi_rows = []
for seg in seg_order:
    if seg not in seg_profile.index:
        continue
    params   = campaign_params[seg]
    n_cust   = seg_profile.loc[seg, 'n_customers']
    avg_clv  = seg_profile.loc[seg, 'avg_clv_12m']
    budget   = params['budget_per_customer']
    conv     = params['conversion_rate']

    total_cost    = budget * n_cust
    expected_rev  = avg_clv * conv * n_cust
    net_roi       = expected_rev - total_cost
    roi_pct       = (net_roi / total_cost * 100) if total_cost > 0 else float('inf')

    roi_rows.append({
        'Segment':             seg,
        'Customers':           n_cust,
        'Budget/Customer':     f'${budget}',
        'Conversion Rate':     f'{conv:.0%}',
        'Avg CLV':             f'${avg_clv:,.2f}',
        'Total Campaign Cost': f'${total_cost:,.0f}',
        'Expected Revenue':    f'${expected_rev:,.0f}',
        'Net ROI':             f'${net_roi:,.0f}',
        'ROI %':               f'{roi_pct:.0f}%' if total_cost > 0 else 'N/A (no spend)',
        'Action':              params['action'],
    })

roi_df = pd.DataFrame(roi_rows)
print("=== Campaign ROI Allocation Table ===")
print(roi_df.to_string(index=False))

### 4. Discussion: Interpreting Campaign ROI

If campaigns show negative ROI for some segments, the model is still valuable for:

1. **Identifying High Value customers to protect margin.** These customers purchase organically at high rates. The model identifies them so we avoid eroding their margin with unnecessary discounts.

2. **Prioritizing budget allocation across segments.** Even when no segment produces positive campaign ROI, the model tells us *where* each marketing dollar has the highest marginal return. Spending $15/customer on Growing is better than spending $15/customer on At-Risk.

3. **Setting data-driven budget caps per segment.** Negative ROI means the assumed budget/customer exceeds the expected incremental value. The model provides the ceiling: if Growing has avg CLV of $X and 25% conversion, then break-even budget is $X * 0.25 per customer. This replaces gut-feel budgeting with quantitative guardrails.

4. **Measuring what matters.** The campaign parameters above (budget, conversion rate) are *assumptions*. The CLV model is the *measurement*. When real campaign data becomes available, these assumptions get replaced with actuals, and the ROI table becomes a true P&L.

### 5. Save Final Outputs

In [ ]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/clv_final.csv', index=False)
print(f"Final CLV data saved to data/processed/clv_final.csv")
print(f"  Shape: {df.shape[0]:,} customers x {df.shape[1]} columns")
print(f"  Key columns: segment, clv_12m, p_purchase, spend_tier")

In [ ]:
seg_order = ['High Value', 'Growing', 'At-Risk', 'Low Value']

print("=" * 55)
print("CLV PIPELINE COMPLETE")
print("=" * 55)
print(f"Total customers:      {len(df):,}")
print(f"Total predicted CLV:  ${df['clv_12m'].sum():,.0f}")
print(f"Median CLV:           ${df['clv_12m'].median():.2f}")
print()
for seg in seg_order:
    seg_data = df[df['segment'] == seg]
    if len(seg_data) > 0:
        n = len(seg_data)
        rev = seg_data['clv_12m'].sum()
        pct = rev / df['clv_12m'].sum() * 100
        print(f"  {seg:15s}: {n:,} customers | ${rev:>10,.0f} predicted ({pct:.1f}%)")
print("=" * 55)
print("\nOutput: data/processed/clv_final.csv")
print("Dashboard: streamlit run src/app.py")